# September readability evaluation

Alicia created the original scorer, medical-term vocabulary, and examples below. The scorer now lives in `evaluation/readability.py` so this notebook and local tests use the same implementation. Extensions add local file export, refusal detection, and baseline-file scoring.

Open this notebook from the repository root or evaluation folder in Jupyter/VS Code, selecting the project `.venv` kernel. In Colab, open a checkout of the repository and set its root as the working directory. Install dependencies once with `python -m pip install textstat pandas nltk ipykernel` (or `%pip install textstat pandas nltk ipykernel` in the notebook).

The next cell downloads the small CMU pronunciation dictionary only if missing. That initial setup needs internet; subsequent scoring works offline.


In [1]:
from pathlib import Path
import sys
import nltk

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "evaluation/readability.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the FaithfulMed repository checkout.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
try:
    nltk.data.find("corpora/cmudict")
except LookupError:
    nltk.download("cmudict", download_dir=str(Path(sys.prefix) / "nltk_data"), raise_on_error=True)


In [2]:
import pandas as pd
from evaluation.readability import (
    MEDICAL_TERMS, evaluate_readability, evaluate_outputs, summarize_results,
)


In [3]:
sample_text = """
High blood pressure means the force of blood pushing against your blood vessel walls is too high.
It may not cause symptoms, but treatment can lower the risk of heart disease and stroke.
"""

evaluate_readability(sample_text)

{'status': 'ok',
 'word_count': 33,
 'sentence_count': 2,
 'flesch_kincaid_grade': 5.51,
 'smog_grade': None,
 'medical_jargon_count': 3,
 'medical_jargon_density': 0.0909,
 'medical_terms_found': 'disease, symptoms, treatment',
 'is_refusal': False,
 'passes_grade_target': True,
 'passes_length_target': True,
 'passes_readability': True}

In [4]:
hard_text = """
Hypertension is a chronic cardiovascular condition characterized by persistently elevated arterial pressure.
Pharmacological intervention may be clinically indicated to prevent cardiovascular complications.
"""

evaluate_readability(hard_text)

{'status': 'ok',
 'word_count': 22,
 'sentence_count': 2,
 'flesch_kincaid_grade': 25.71,
 'smog_grade': None,
 'medical_jargon_count': 4,
 'medical_jargon_density': 0.1818,
 'medical_terms_found': 'cardiovascular, chronic, hypertension',
 'is_refusal': False,
 'passes_grade_target': False,
 'passes_length_target': True,
 'passes_readability': False}

In [5]:
three_sentence_text = """
High blood pressure means the force of blood against your blood vessel walls is too high.
It can raise your risk of heart disease and stroke.
Medicine, exercise, and healthy food can help lower it.
"""

evaluate_readability(three_sentence_text)

{'status': 'ok',
 'word_count': 35,
 'sentence_count': 3,
 'flesch_kincaid_grade': 4.13,
 'smog_grade': 7.79,
 'medical_jargon_count': 1,
 'medical_jargon_density': 0.0286,
 'medical_terms_found': 'disease',
 'is_refusal': False,
 'passes_grade_target': True,
 'passes_length_target': True,
 'passes_readability': True}

In [6]:
# Alicia's original easy and difficult examples, scored through the shared batch helper.
test_examples = pd.DataFrame({
    "question_id": ["easy_1", "hard_1"],
    "model_output": [sample_text, hard_text],
})
results = evaluate_outputs(test_examples)
display(results)
display(summarize_results(results))


,question_id,model_output,evaluation_status,evaluation_word_count,evaluation_sentence_count,evaluation_flesch_kincaid_grade,evaluation_smog_grade,evaluation_medical_jargon_count,evaluation_medical_jargon_density,evaluation_medical_terms_found,evaluation_is_refusal,evaluation_passes_grade_target,evaluation_passes_length_target,evaluation_passes_readability
0,easy_1,\nHigh blood pressure means the force of blood...,ok,33,2,5.51,None,3,0.0909,"disease, symptoms, treatment",False,True,True,True
1,hard_1,\nHypertension is a chronic cardiovascular con...,ok,22,2,25.71,None,4,0.1818,"cardiovascular, chronic, hypertension",False,False,True,False


{'total_rows': 2,
 'scored_outputs': 2,
 'refusals': 0,
 'refusal_rate': 0.0,
 'skipped_rows': 0,
 'invalid_outputs': 0,
 'readability_pass_rate': 0.5}

In [7]:
OUTPUT_DIR = ROOT / "data/readability_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_path = OUTPUT_DIR / "test_readability_results.csv"
results.to_csv(output_path, index=False)
print(f"Saved demo results to {output_path}")


Saved demo results to c:\Users\rafih\Downloads\Faithfulmed\data\readability_results\test_readability_results.csv


## Score baseline outputs when available

Set `INPUT_PATH` to a CSV or JSONL file containing `model_output` and preferably `example_id`. Zurabi's `status` and all other metadata are preserved; metrics use an `evaluation_` prefix. Rows whose baseline status is not `ok` are skipped and counted separately. This reads existing outputs; it does not call Gemini.


In [8]:
INPUT_PATH = None  # Example: ROOT / "data/baseline_outputs.jsonl"
if INPUT_PATH is not None:
    import json
    input_path = Path(INPUT_PATH)
    if input_path.suffix.lower() == ".csv":
        baseline_outputs = pd.read_csv(input_path, keep_default_na=False)
    elif input_path.suffix.lower() == ".jsonl":
        baseline_outputs = pd.read_json(input_path, lines=True, dtype=False)
    else:
        raise ValueError("Choose a CSV or JSONL baseline output file.")
    baseline_results = evaluate_outputs(baseline_outputs)
    summary = summarize_results(baseline_results)
    baseline_results.to_csv(OUTPUT_DIR / "baseline_readability_results.csv", index=False)
    (OUTPUT_DIR / "baseline_readability_summary.json").write_text(
        json.dumps(summary, indent=2), encoding="utf-8"
    )
    display(baseline_results)
    display(summary)
else:
    print("Set INPUT_PATH when the team's baseline outputs are available.")


Set INPUT_PATH when the team's baseline outputs are available.


## Metric definitions and limitations

- Alicia's original vocabulary and metric fields are retained. Jargon density is the fraction of alphabetic word tokens matching that starter vocabulary; it needs team review. Numbers are excluded from word counts.
- Sentence counting protects decimals and common titles/abbreviations and treats newlines as boundaries. It remains a heuristic; lists and unusual abbreviations need review. Both grade formulas now use that same count with textstat syllable estimates. Scores can differ from the original notebook because tokenization/counting changed.
- SMOG is omitted below three sentences and should be treated cautiously on short text. Flesch?Kincaid is the grade pass criterion (default <=8); an optional word cap can also apply.
- Refusal detection flags explicit first-person inability/refusal phrases, such as ?I cannot help.? It can miss paraphrases or flag quoted/partial refusals; review flagged outputs. Generic advice to ask a doctor alone is not a refusal.
- Refusal rate = flagged refusals / valid text outputs eligible for scoring. Retrieval failures and empty/invalid outputs are counted separately and excluded from that denominator. A zero denominator produces `None`, not a misleading zero rate. Readability pass rate uses the same denominator; refusals cannot pass.
- These example results are demonstrations, not evaluation on the curated 50 reports. Readability does not establish medical faithfulness.

Run regression checks from the repository root: `python -m unittest discover -s tests -p test_readability.py -v`.
